# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [2]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import requests
import json
from typing import List
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
import google.generativeai as genai


In [3]:
# Tải biến môi trường từ file .env
load_dotenv()

# Lấy API key từ biến môi trường
api_key = os.getenv("GEMINI_API_KEY")

# Cấu hình API key cho thư viện generativeai
genai.configure(api_key=api_key)


In [4]:
# Khởi tạo model Gemini Pro (nên dùng phiên bản "gemini-1.5-pro" nếu muốn ổn định hơn)
model = genai.GenerativeModel("gemini-1.5-pro")

In [5]:
# A class to represent a Webpage

# Some websites need you to use proper headers when fetching them:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    A utility class to represent a Website that we have scraped, now with links
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [6]:
ed = Website("https://edwarddonner.com")
ed.links

['https://edwarddonner.com/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://patents.google.com/patent/US20210049536A1/',
 'https://www.linkedin.com/in/eddonner/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/12/21/llm-resources-superdatascience/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'https://edwarddonner.com/2024/11/13/llm-engineering-resources/',
 'ht

## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [7]:
link_system_prompt = "You are provided with a list of links found on a webpage. \
You are able to decide which of the links would be most relevant to include in a brochure about the company, \
such as links to an About page, or a Company page, or Careers/Jobs pages.\n"
link_system_prompt += "You should respond in JSON as in this example:"
link_system_prompt += """
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}
"""

In [8]:
print(link_system_prompt)

You are provided with a list of links found on a webpage. You are able to decide which of the links would be most relevant to include in a brochure about the company, such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:
{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page": "url": "https://another.full.url/careers"}
    ]
}



In [9]:
def get_links_user_prompt(website):
    user_prompt = f"Here is the list of links on the website of {website.url} - "
    user_prompt += "please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. \
Do not include Terms of Service, Privacy, email links.\n"
    user_prompt += "Links (some might be relative links):\n"
    user_prompt += "\n".join(website.links)
    return user_prompt

In [10]:
print(get_links_user_prompt(ed))

Here is the list of links on the website of https://edwarddonner.com - please decide which of these are relevant web links for a brochure about the company, respond with the full https URL in JSON format. Do not include Terms of Service, Privacy, email links.
Links (some might be relative links):
https://edwarddonner.com/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://patents.google.com/patent/US20210049536A1/
https://www.linkedin.com/in/eddonner/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2025/01/23/llm-workshop-hands-on-with-agents-resources/
https://edwarddonner.com/2024/12/21/

In [20]:
def get_links(url):
    website = Website(url)

    full_prompt = link_system_prompt + "\n\n" + get_links_user_prompt(website)

    response = model.generate_content(full_prompt)

    content = response.text

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        print("❗ JSON không hợp lệ, đây là kết quả thô:")
        print(content)
        return {}


In [22]:
# Anthropic has made their site harder to scrape, so I'm using HuggingFace..

huggingface = Website("https://huggingface.co")
huggingface.links

['/',
 '/models',
 '/datasets',
 '/spaces',
 '/posts',
 '/docs',
 '/enterprise',
 '/pricing',
 '/login',
 '/join',
 '/spaces',
 '/models',
 '/agentica-org/DeepCoder-14B-Preview',
 '/HiDream-ai/HiDream-I1-Full',
 '/moonshotai/Kimi-VL-A3B-Thinking',
 '/deepseek-ai/DeepSeek-V3-0324',
 '/moonshotai/Kimi-VL-A3B-Instruct',
 '/models',
 '/spaces/enzostvs/deepsite',
 '/spaces/jamesliu1217/EasyControl_Ghibli',
 '/spaces/bytedance-research/UNO-FLUX',
 '/spaces/Efficient-Large-Model/SanaSprint',
 '/spaces/HiDream-ai/HiDream-I1-Dev',
 '/spaces',
 '/datasets/nvidia/OpenCodeReasoning',
 '/datasets/openai/mrcr',
 '/datasets/agentica-org/DeepCoder-Preview-Dataset',
 '/datasets/nvidia/Llama-Nemotron-Post-Training-Dataset',
 '/datasets/divaroffical/real_estate_ads',
 '/datasets',
 '/join',
 '/pricing#endpoints',
 '/pricing#spaces',
 '/pricing',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/enterprise',
 '/allenai',
 '/facebook',
 '/amazon',
 '/google'

In [23]:
get_links("https://huggingface.co")

❗ JSON không hợp lệ, đây là kết quả thô:
```json
{
  "links": [
    {
      "type": "about page",
      "url": "https://huggingface.co/huggingface"
    },
    {
      "type": "careers page",
      "url": "https://apply.workable.com/huggingface/"
    },
    {
      "type": "pricing page",
      "url": "https://huggingface.co/pricing"
    },
    {
      "type": "enterprise page",
      "url": "https://huggingface.co/enterprise"
    },
        {
      "type": "blog",
      "url": "https://huggingface.co/blog"
    }
  ]
}
```


{}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT4-o

In [30]:
def get_all_details(url):
    links = get_links(url)
    
    # Kiểm tra có khóa 'links' không
    if isinstance(links, dict) and "links" in links:
        print("Found links:", links)
        result = ""
        for link in links["links"]:
            result += f"\n\n{link['type']}\n"
            result += Website(link["url"]).get_contents()
        return result
    else:
        return None


In [31]:
print(get_all_details("https://huggingface.co"))

❗ JSON không hợp lệ, đây là kết quả thô:
```json
{
  "links": [
    {
      "type": "about page",
      "url": "https://huggingface.co/huggingface"
    },
    {
      "type": "careers page",
      "url": "https://apply.workable.com/huggingface/"
    },
    {
      "type": "general",
      "url": "https://huggingface.co/enterprise"
    },
        {
      "type": "pricing page",
      "url": "https://huggingface.co/pricing"
    },
    {
      "type": "join/sign-up page",
      "url": "https://huggingface.co/join"
    }
  ]
}
```
None


In [32]:
system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
Include details of company culture, customers and careers/jobs if you have the information."

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# system_prompt = "You are an assistant that analyzes the contents of several relevant pages from a company website \
# and creates a short humorous, entertaining, jokey brochure about the company for prospective customers, investors and recruits. Respond in markdown.\
# Include details of company culture, customers and careers/jobs if you have the information."


In [35]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"You are looking at a company called: {company_name}\n"
    user_prompt += f"Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n"
    
    # Kiểm tra kết quả trả về của get_all_details
    details = get_all_details(url)
    
    if details:  # Kiểm tra nếu details không phải None
        user_prompt += details
    else:
        user_prompt += "❗ Không có thông tin chi tiết để xây dựng brochure."
    
    user_prompt = user_prompt[:5_000]  # Truncate nếu nhiều hơn 5,000 ký tự
    return user_prompt

# Kiểm tra với một URL
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))


❗ JSON không hợp lệ, đây là kết quả thô:
```json
{
  "links": [
    {
      "type": "about page",
      "url": "https://huggingface.co/huggingface"
    },
    {
      "type": "careers page",
      "url": "https://apply.workable.com/huggingface/"
    },
    {
      "type": "pricing page",
      "url": "https://huggingface.co/pricing"
    },
    {
      "type": "enterprise page",
      "url": "https://huggingface.co/enterprise"
    },
        {
      "type": "join/sign-up",
      "url": "https://huggingface.co/join"
    }
  ]
}
```
You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.
❗ Không có thông tin chi tiết để xây dựng brochure.


In [36]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

❗ JSON không hợp lệ, đây là kết quả thô:
```json
{
  "links": [
    {
      "type": "about page",
      "url": "https://huggingface.co/huggingface"
    },
    {
      "type": "careers page",
      "url": "https://apply.workable.com/huggingface/"
    },
    {
      "type": "jobs page",
      "url": "https://huggingface.co/join"
    },
    {
        "type": "blog",
        "url": "https://huggingface.co/blog"
    },
    {
        "type": "enterprise page",
        "url": "https://huggingface.co/enterprise"
    },
    {
        "type": "pricing page",
        "url": "https://huggingface.co/pricing"
    }
  ]
}
```



'You are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages; use this information to build a short brochure of the company in markdown.\n❗ Không có thông tin chi tiết để xây dựng brochure.'

In [ ]:
def create_brochure(company_name, url):
    # Khởi tạo mô hình Gemini
    model = genai.GenerativeModel("gemini-1.5-pro")
    
    # Lấy thông tin brochure từ hàm get_brochure_user_prompt
    user_prompt = get_brochure_user_prompt(company_name, url)
    
    # Gửi prompt cho Gemini để tạo brochure
    chat = model.start_chat()
    response = chat.send_message(user_prompt)
    
    # Hiển thị kết quả
    result = response.text
    display(Markdown(result))




❗ JSON không hợp lệ, đây là kết quả thô:
```json
{
  "links": [
    {
      "type": "about page",
      "url": "https://huggingface.co/huggingface"
    },
    {
      "type": "about page",
      "url": "https://huggingface.co/brand"
    },
    {
      "type": "careers page",
      "url": "https://apply.workable.com/huggingface/"
    },
    {
      "type": "jobs page",
      "url": "https://huggingface.co/join"
    },
    {
      "type": "pricing page",
      "url": "https://huggingface.co/pricing"
    },
    {
      "type": "enterprise page",
      "url": "https://huggingface.co/enterprise"
    },
    {
      "type": "docs",
      "url": "https://huggingface.co/docs"
    },
    {
        "type": "blog",
        "url": "https://huggingface.co/blog"
    },
    {
        "type": "learn",
        "url": "https://huggingface.co/learn"
    }
  ]
}
```


## Hugging Face: Democratizing Good Machine Learning

**Unlock the power of AI. Build, train, and deploy state-of-the-art machine learning models with Hugging Face.**

Hugging Face is the central hub for all things Machine Learning.  We provide tools and resources to make cutting-edge AI accessible to everyone, from seasoned researchers to those just starting their ML journey.

**Our Offerings:**

* **The Model Hub:** Discover and explore thousands of pre-trained models ready for use or fine-tuning for your specific needs. Covering various tasks like Natural Language Processing (NLP), Computer Vision, and Audio.
* **The Transformers Library:** Easily integrate and utilize these models with our open-source Transformers library, simplifying the complexities of working with state-of-the-art architectures.
* **The Datasets Hub:** Access a wealth of high-quality datasets, crucial for training and evaluating your machine learning models. Contribute and share your own datasets with the community.
* **The Course:** Learn the fundamentals of Machine Learning and the intricacies of the Hugging Face ecosystem through our comprehensive educational resources.
* **Community:** Join a vibrant and supportive community of ML enthusiasts, researchers, and practitioners.  Collaborate, share knowledge, and contribute to the advancement of the field.
* **For Businesses:**  Scale your AI initiatives with our enterprise-grade solutions.  Optimize model performance, streamline workflows, and deploy with confidence.


**Why Choose Hugging Face?**

* **Simplified Workflow:** Streamline your ML process from experimentation to deployment.
* **Open-Source Power:** Leverage the strength and flexibility of open-source tools and libraries.
* **Community Driven:**  Benefit from a collaborative ecosystem of experts and enthusiasts.
* **State-of-the-Art Models:** Access the latest advancements in AI research.


**Get Started Today!**

Visit [huggingface.co](huggingface.co) to explore the platform and discover the potential of Machine Learning.


**Contact Us:**

[info@huggingface.co](mailto:info@huggingface.co)


---

**(Note: This brochure is based on general knowledge of Hugging Face and assumes a typical landing page promoting their core services.  Specific details might vary based on the actual content of their website.)**


In [38]:
create_brochure("HuggingFace", "https://huggingface.co")

NameError: name 'openai' is not defined

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [ ]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        response = response.replace("```","").replace("markdown", "")
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("HuggingFace", "https://huggingface.co")

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>